<a href="https://colab.research.google.com/github/JozefSL/pyNotes/blob/main/numpy/RegMprod_wIP_vector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [23]:
import numpy as np
import pandas as pd
from IPython.display import display, HTML

# The generate_arps_profile function is not used for the new well production calculation.

# Define the start date as a variable
START_DATE_STR = '2023-12-01' # This represents the M0 month, e.g., 'YYYY-MM-DD'
timeline_length = 24 # Number of months to forecast



# 1. Load the CSV files
df = pd.read_csv("CurveData.csv")
factor_df = pd.read_csv("FactorDataT.csv")
well_count_df = pd.read_csv("WellCount.csv")

def generate_base_schedule_matrix(type_curve, num_months):
    """
    Generates a reusable base schedule matrix where each row represents a well vintage
    (the month a batch of wells is turned online) and each column represents a calendar timeline month.

    Parameters:
    type_curve (np.ndarray): The absolute production profile vector for a single standard well.
    timeline_months (int): Total number of months to model in the forecast timeline.

    Returns:
    np.ndarray: A 2D matrix of shape (timeline_months, timeline_months) filled with shifted type curves.
    """
    base_matrix = np.zeros((num_months, num_months))

    for start_month in range(num_months):
        # Calculate how many months are left in the forecast timeline from this start month forward
        months_remaining = num_months - start_month

        # Determine how much of the type curve fits into the remaining timeline window
        curve_length = min(len(type_curve), months_remaining)

        # Populate the row starting from the diagonal (the month the well goes online)
        base_matrix[start_month, start_month:start_month + curve_length] = type_curve[:curve_length]

    return base_matrix



# Identify our timeline_length in months for the new decline curve (M1 through Mxx)
# Corrected to use uppercase 'M' for column names to match the CSV data
month_cols = [f'M{i+1}' for i in range(timeline_length)]
num_months = len(month_cols)

# Dictionary to hold the final forecast timelines for analysis
regional_forecasts = {}

# 2. Process data dynamically by Region for new well production
for region, group in df.groupby('Region'):
    # Extract the new well decline curve (normalized) for the current region
    # Convert non-numeric values to 0 and default to zeros if 'newWellDC' is not found
    new_well_decline_curve_values = np.zeros(num_months)
    legacy_decline_curve_values = np.zeros(num_months)
    new_well_decline_curve_row = group.loc[group['Name'] == 'newWellDC', month_cols]
    legacy_decline_curve_row = group.loc[group['Name'] == 'legacyDC', month_cols]
    if not new_well_decline_curve_row.empty:
        new_well_decline_curve_values = pd.to_numeric(new_well_decline_curve_row.iloc[0], errors='coerce').fillna(0).values
    if not legacy_decline_curve_row.empty:
        legacy_decline_curve_values = pd.to_numeric(legacy_decline_curve_row.iloc[0], errors='coerce').fillna(0).values

    # Get IP rate for the current region from FactorDataT.csv
    # Default to 0.0 if not found for the region
    ip_rate = 0.0
    ip_rate_row = factor_df.loc[factor_df['Region'] == region, 'P1']
    if not ip_rate_row.empty:
        ip_rate = ip_rate_row.values[0]
    # Adjust normalized new-well decline values with IP rate for the current region
    new_well_decline_curve_values = new_well_decline_curve_values * ip_rate

    # Get Well Count *vector* for the current region from WellCount.csv
    # This should be a vector of well counts for M1 to M24, where each element corresponds
    # to the well count for the vintage starting in that month.
    well_count_vector = np.zeros(num_months)
    well_count_row_full = well_count_df.loc[(well_count_df['Region'] == region) & (well_count_df['Name'] == 'wellCount'), month_cols]
    if not well_count_row_full.empty:
        well_count_vector = pd.to_numeric(well_count_row_full.iloc[0], errors='coerce').fillna(0).values

    # Generate the reusable Base Schedule Matrix template (completely independent of well counts)
    base_prod_matrix = generate_base_schedule_matrix(new_well_decline_curve_values, num_months)

    # Scale the base unit matrix by the scenario's well count column (well_count_vector) using broadcasting
    new_well_production_matrix = base_prod_matrix * well_count_vector[:, np.newaxis]


    # Get M0 production rate for the current region from FactorDataT.csv
    # Default to 0.0 if not found for the region
    m0_value = 0.0 # Initialize scalar M0
    m0_series_row = factor_df.loc[factor_df['Region'] == region, 'M0']*1000
    if not m0_series_row.empty:
        # Extract scalar value from the series for both calculation and prepending
        m0_value = pd.to_numeric(m0_series_row, errors='coerce').fillna(0).iloc[0]

    # Adjust normalized legacy decline values with M0 rate for the current region
    # Use the scalar m0_value for multiplication
    legacy_decline_curve_values = legacy_decline_curve_values * m0_value

    # Sum columns to get total timeline production for this region (timeline_length months)
    current_regional_forecast = legacy_decline_curve_values + np.sum(new_well_production_matrix, axis=0)

    # Prepend the M0 value and timeline_length months
    regional_forecasts[region] = np.insert(current_regional_forecast, 0, m0_value)





# 3. Combine results into a clean summary DataFrame
# Convert START_DATE_STR to datetime object
start_date_dt = pd.to_datetime(START_DATE_STR)

# Define the new index labels including the dynamic start month
initial_month_label = start_date_dt.strftime('%b_%Y')

# Generate dates for the subsequent months
dates_after_initial_month = pd.date_range(start=start_date_dt + pd.DateOffset(months=1), periods=num_months, freq='MS')
formatted_dates = [d.strftime('%b_%Y') for d in dates_after_initial_month]

new_index_labels = [initial_month_label] + formatted_dates

forecast_df = pd.DataFrame(regional_forecasts, index=new_index_labels)
forecast_df['L48'] = forecast_df.sum(axis=1)
forecast_df = forecast_df.astype(float)

print("Primary Commodity Forecast Summary:")

display(forecast_df.style.format("{:,.0f}"))
#print(forecast_df)

Primary Commodity Forecast Summary:


,Appalachia,Bakken,EagleFord,Haynesville,Permian,R48,L48
Dec_2023,"36,722,000","1,305,000","1,068,000","16,014,000","6,193,000","2,096,000","63,398,000"
Jan_2024,"36,625,105","1,297,516","1,069,418","15,973,921","6,195,689","2,086,555","63,248,204"
Feb_2024,"36,289,845","1,289,165","1,081,109","15,906,978","6,164,006","2,069,465","62,800,568"
Mar_2024,"35,850,213","1,272,933","1,112,342","15,571,866","6,140,300","2,060,448","62,008,103"
Apr_2024,"35,585,661","1,248,740","1,143,298","15,124,817","6,163,037","2,054,583","61,320,136"
May_2024,"35,464,388","1,234,513","1,151,076","14,794,622","6,250,850","2,047,794","60,943,243"
Jun_2024,"35,174,954","1,223,975","1,135,046","14,564,796","6,304,278","2,034,106","60,437,155"
Jul_2024,"35,045,531","1,215,485","1,145,498","14,501,625","6,309,440","2,019,357","60,236,936"
Aug_2024,"34,870,240","1,206,912","1,164,703","14,320,855","6,304,329","2,018,783","59,885,821"
Sep_2024,"34,499,564","1,211,461","1,173,309","13,998,651","6,291,594","2,023,562","59,198,142"


### Non-Primary Commodity Forecast

Now, let's calculate the non-primary commodity production (e.g., gas for oil regions, oil for gas regions) using the GOR (Gas-Oil Ratio) and GOR growth factors from `FactorDataT.csv`.

In [24]:
# Dictionary to hold non-primary commodity forecasts
non_primary_regional_forecasts = {}

# Define primary commodity types for each region
# Appalachia and Haynesville are gas (Mcf/d), others are oil (bbl/d)
GAS_REGIONS = ['Appalachia', 'Haynesville']

# Iterate through each region to calculate non-primary commodity production
for region, forecast_values in regional_forecasts.items():
    # Get GOR for M0 and gorGrowth for the current region
    # Default to 0 if not found
    gor_m0 = 0.0
    gor_growth = 0.0 # Assuming this is a percentage (e.g., 0.5 for 0.5% growth)

    gor_row = factor_df.loc[factor_df['Region'] == region]
    if not gor_row.empty:
        gor_m0 = pd.to_numeric(gor_row['GOR'], errors='coerce').fillna(0).iloc[0]
        # Assuming gorGrowth is a monthly percentage, so convert to a multiplier
        gor_monthly_growth = pd.to_numeric(gor_row['gorGrowth'], errors='coerce').fillna(0).iloc[0]/timeline_length
    else:
        gor_monthly_growth = 0.0 # No growth if no data

    # Calculate monthly GOR values for the timeline length
    monthly_gor_values = np.zeros(timeline_length+1)
    monthly_gor_values[0] = gor_m0
    for i in range(1, timeline_length+1):
        monthly_gor_values[i] = gor_m0 + i * gor_monthly_growth

    # Derive non-primary production based on the primary commodity
    non_primary_production = np.zeros(timeline_length+1)
    if region in GAS_REGIONS: # Primary is gas, calculate oil
        # Oil (bbl/d) = Gas (Mcf/d) / GOR (Mcf/bbl)
        # Avoid division by zero if GOR is 0
        non_primary_production = np.where(monthly_gor_values != 0, forecast_values / monthly_gor_values, 0)
    else: # Primary is oil, calculate gas
        # Gas (Mcf/d) = Oil (bbl/d) * GOR (Mcf/bbl)
        non_primary_production = forecast_values * monthly_gor_values

    non_primary_regional_forecasts[region] = non_primary_production

# Create a DataFrame for non-primary commodity forecasts
non_primary_forecast_df = pd.DataFrame(non_primary_regional_forecasts, index=new_index_labels)
non_primary_forecast_df['L48'] = non_primary_forecast_df.sum(axis=1)
non_primary_forecast_df = non_primary_forecast_df.astype(float)

print("Primary Commodity Forecast (bbl/day or Mcf/day based on region):")
display(forecast_df.style.format("{:,.0f}"))

print("\nNon-Primary Commodity Forecast (bbl/day or Mcf/day based on region):")
display(non_primary_forecast_df.style.format("{:,.0f}"))

Primary Commodity Forecast (bbl/day or Mcf/day based on region):


,Appalachia,Bakken,EagleFord,Haynesville,Permian,R48,L48
Dec_2023,"36,722,000","1,305,000","1,068,000","16,014,000","6,193,000","2,096,000","63,398,000"
Jan_2024,"36,625,105","1,297,516","1,069,418","15,973,921","6,195,689","2,086,555","63,248,204"
Feb_2024,"36,289,845","1,289,165","1,081,109","15,906,978","6,164,006","2,069,465","62,800,568"
Mar_2024,"35,850,213","1,272,933","1,112,342","15,571,866","6,140,300","2,060,448","62,008,103"
Apr_2024,"35,585,661","1,248,740","1,143,298","15,124,817","6,163,037","2,054,583","61,320,136"
May_2024,"35,464,388","1,234,513","1,151,076","14,794,622","6,250,850","2,047,794","60,943,243"
Jun_2024,"35,174,954","1,223,975","1,135,046","14,564,796","6,304,278","2,034,106","60,437,155"
Jul_2024,"35,045,531","1,215,485","1,145,498","14,501,625","6,309,440","2,019,357","60,236,936"
Aug_2024,"34,870,240","1,206,912","1,164,703","14,320,855","6,304,329","2,018,783","59,885,821"
Sep_2024,"34,499,564","1,211,461","1,173,309","13,998,651","6,291,594","2,023,562","59,198,142"



Non-Primary Commodity Forecast (bbl/day or Mcf/day based on region):


,Appalachia,Bakken,EagleFord,Haynesville,Permian,R48,L48
Dec_2023,"131,150","3,523,500","6,728,400","26,690","24,772,000","26,200,000","61,381,740"
Jan_2024,"132,780","3,519,513","6,768,525","26,586","24,911,834","26,125,414","61,484,651"
Feb_2024,"133,582","3,512,974","6,874,051","26,438","24,912,856","25,954,545","61,414,446"
Mar_2024,"134,019","3,484,655","7,105,087","25,845","24,944,971","25,884,377","61,578,955"
Apr_2024,"135,135","3,434,035","7,336,165","25,069","25,165,736","25,853,499","61,949,639"
May_2024,"136,840","3,410,343","7,419,645","24,488","25,654,529","25,810,737","62,456,581"
Jun_2024,"137,941","3,396,531","7,349,422","24,074","26,005,148","25,680,588","62,593,705"
Jul_2024,"139,716","3,388,164","7,450,512","23,937","26,157,887","25,536,451","62,696,667"
Aug_2024,"141,366","3,379,354","7,609,391","23,606","26,268,038","25,571,246","62,993,001"
Sep_2024,"142,266","3,407,234","7,699,842","23,043","26,346,052","25,673,939","63,292,376"


### Combined Oil and Gas Forecasts

Let's combine the primary and non-primary commodity forecasts into distinct Oil and Gas production dataframes.

In [25]:
oil_forecast_combined = {}
gas_forecast_combined = {}

# Assuming GAS_REGIONS is already defined in a previous cell
# GAS_REGIONS = ['Appalachia', 'Haynesville']

for region in regional_forecasts.keys(): # Iterate through all regions
    if region in GAS_REGIONS: # Primary is Gas, Non-primary is Oil
        gas_forecast_combined[region] = forecast_df[region].values
        oil_forecast_combined[region] = non_primary_forecast_df[region].values
    else: # Primary is Oil, Non-primary is Gas
        oil_forecast_combined[region] = forecast_df[region].values
        gas_forecast_combined[region] = non_primary_forecast_df[region].values

# Create combined Oil and Gas DataFrames
oil_forecast_combined_df = pd.DataFrame(oil_forecast_combined, index=new_index_labels)
gas_forecast_combined_df = pd.DataFrame(gas_forecast_combined, index=new_index_labels)

# Add 'Total Company' (L48) column to both combined dataframes
oil_forecast_combined_df['L48'] = oil_forecast_combined_df.sum(axis=1)
gas_forecast_combined_df['L48'] = gas_forecast_combined_df.sum(axis=1)

# Ensure data types are float
oil_forecast_combined_df = oil_forecast_combined_df.astype(float)
gas_forecast_combined_df = gas_forecast_combined_df.astype(float)

print("\n--- Final Oil Production Forecast (bbl/day) ---")
display(oil_forecast_combined_df.style.format("{:,.0f}"))

print("\n--- Final Gas Production Forecast (Mcf/day) ---")
display(gas_forecast_combined_df.style.format("{:,.0f}"))


--- Final Oil Production Forecast (bbl/day) ---


,Appalachia,Bakken,EagleFord,Haynesville,Permian,R48,L48
Dec_2023,"131,150","1,305,000","1,068,000","26,690","6,193,000","2,096,000","10,819,840"
Jan_2024,"132,780","1,297,516","1,069,418","26,586","6,195,689","2,086,555","10,808,545"
Feb_2024,"133,582","1,289,165","1,081,109","26,438","6,164,006","2,069,465","10,763,765"
Mar_2024,"134,019","1,272,933","1,112,342","25,845","6,140,300","2,060,448","10,745,889"
Apr_2024,"135,135","1,248,740","1,143,298","25,069","6,163,037","2,054,583","10,769,863"
May_2024,"136,840","1,234,513","1,151,076","24,488","6,250,850","2,047,794","10,845,561"
Jun_2024,"137,941","1,223,975","1,135,046","24,074","6,304,278","2,034,106","10,859,421"
Jul_2024,"139,716","1,215,485","1,145,498","23,937","6,309,440","2,019,357","10,853,433"
Aug_2024,"141,366","1,206,912","1,164,703","23,606","6,304,329","2,018,783","10,859,698"
Sep_2024,"142,266","1,211,461","1,173,309","23,043","6,291,594","2,023,562","10,865,236"



--- Final Gas Production Forecast (Mcf/day) ---


,Appalachia,Bakken,EagleFord,Haynesville,Permian,R48,L48
Dec_2023,"36,722,000","3,523,500","6,728,400","16,014,000","24,772,000","26,200,000","113,959,900"
Jan_2024,"36,625,105","3,519,513","6,768,525","15,973,921","24,911,834","26,125,414","113,924,310"
Feb_2024,"36,289,845","3,512,974","6,874,051","15,906,978","24,912,856","25,954,545","113,451,248"
Mar_2024,"35,850,213","3,484,655","7,105,087","15,571,866","24,944,971","25,884,377","112,841,169"
Apr_2024,"35,585,661","3,434,035","7,336,165","15,124,817","25,165,736","25,853,499","112,499,913"
May_2024,"35,464,388","3,410,343","7,419,645","14,794,622","25,654,529","25,810,737","112,554,264"
Jun_2024,"35,174,954","3,396,531","7,349,422","14,564,796","26,005,148","25,680,588","112,171,440"
Jul_2024,"35,045,531","3,388,164","7,450,512","14,501,625","26,157,887","25,536,451","112,080,170"
Aug_2024,"34,870,240","3,379,354","7,609,391","14,320,855","26,268,038","25,571,246","112,019,124"
Sep_2024,"34,499,564","3,407,234","7,699,842","13,998,651","26,346,052","25,673,939","111,625,283"


### GOR Ratios for the Forecasted Window

Let's calculate and display the Gas-Oil Ratios (GOR) for each month in the forecast window, showing how they evolve per region.

In [26]:
gor_ratios_forecast = {}

for region in regional_forecasts.keys():
    gor_m0 = 0.0
    gor_monthly_growth = 0.0

    gor_row = factor_df.loc[factor_df['Region'] == region]
    if not gor_row.empty:
        gor_m0 = pd.to_numeric(gor_row['GOR'], errors='coerce').fillna(0).iloc[0]
        gor_monthly_growth = pd.to_numeric(gor_row['gorGrowth'], errors='coerce').fillna(0).iloc[0] / timeline_length

    monthly_gor_values_for_region = np.zeros(timeline_length + 1)
    monthly_gor_values_for_region[0] = gor_m0
    for i in range(1, timeline_length + 1):
        monthly_gor_values_for_region[i] = gor_m0 + i * gor_monthly_growth

    gor_ratios_forecast[region] = monthly_gor_values_for_region

gor_ratios_df = pd.DataFrame(gor_ratios_forecast, index=new_index_labels)
gor_ratios_df = gor_ratios_df.astype(float)

print("\n--- GOR Ratios (Mcf/bbl) for Forecasted Window ---")
display(gor_ratios_df.style.format("{:,.2f}"))


--- GOR Ratios (Mcf/bbl) for Forecasted Window ---


,Appalachia,Bakken,EagleFord,Haynesville,Permian,R48
Dec_2023,280.00,2.70,6.30,600.00,4.00,12.50
Jan_2024,275.83,2.71,6.33,600.83,4.02,12.52
Feb_2024,271.67,2.73,6.36,601.67,4.04,12.54
Mar_2024,267.50,2.74,6.39,602.50,4.06,12.56
Apr_2024,263.33,2.75,6.42,603.33,4.08,12.58
May_2024,259.17,2.76,6.45,604.17,4.10,12.60
Jun_2024,255.00,2.78,6.47,605.00,4.12,12.62
Jul_2024,250.83,2.79,6.50,605.83,4.15,12.65
Aug_2024,246.67,2.80,6.53,606.67,4.17,12.67
Sep_2024,242.50,2.81,6.56,607.50,4.19,12.69


### Final Marketed Gas Production Forecast

Now, let's adjust the 'Final Gas Production Forecast' to account for the `ShrinkFactor` for each region, deriving the 'Final Marketed Gas Production Forecast'.

In [27]:
marketed_gas_forecast = {}

for region in gas_forecast_combined_df.columns:
    if region == 'L48': # Skip the total column for this calculation, it will be re-calculated
        continue

    shrink_factor = 1.0 # Default shrink factor if not found
    shrink_row_series = factor_df.loc[factor_df['Region'] == region, 'ShrinkFactor']
    if not shrink_row_series.empty:
        # Convert to numeric and fill any NaN values in the Series before extracting the scalar
        shrink_factor = pd.to_numeric(shrink_row_series, errors='coerce').fillna(1.0).iloc[0]

    # Ensure shrink_factor is not zero to prevent division errors
    if shrink_factor == 0:
        shrink_factor = 1.0

    marketed_gas_forecast[region] = gas_forecast_combined_df[region] * shrink_factor

marketed_gas_forecast_df = pd.DataFrame(marketed_gas_forecast, index=new_index_labels)
marketed_gas_forecast_df['L48'] = marketed_gas_forecast_df.sum(axis=1)
marketed_gas_forecast_df = marketed_gas_forecast_df.astype(float)

print("\n--- Final Marketed Gas Production Forecast (Mcf/day) ---")
display(marketed_gas_forecast_df.style.format("{:,.0f}"))


--- Final Marketed Gas Production Forecast (Mcf/day) ---


,Appalachia,Bakken,EagleFord,Haynesville,Permian,R48,L48
Dec_2023,"36,722,000","3,312,090","6,035,375","15,933,930","23,409,540","25,440,200","110,853,135"
Jan_2024,"36,625,105","3,308,342","6,071,367","15,894,051","23,541,683","25,367,777","110,808,324"
Feb_2024,"36,289,845","3,302,196","6,166,024","15,827,443","23,542,649","25,201,863","110,330,019"
Mar_2024,"35,850,213","3,275,576","6,373,263","15,494,007","23,572,997","25,133,731","109,699,786"
Apr_2024,"35,585,661","3,227,993","6,580,540","15,049,193","23,781,621","25,103,747","109,328,755"
May_2024,"35,464,388","3,205,722","6,655,421","14,720,648","24,243,530","25,062,226","109,351,936"
Jun_2024,"35,174,954","3,192,740","6,592,432","14,491,972","24,574,865","24,935,851","108,962,813"
Jul_2024,"35,045,531","3,184,874","6,683,109","14,429,117","24,719,204","24,795,894","108,857,728"
Aug_2024,"34,870,240","3,176,593","6,825,624","14,249,251","24,823,296","24,829,680","108,774,683"
Sep_2024,"34,499,564","3,202,800","6,906,759","13,928,658","24,897,019","24,929,394","108,364,195"


### Inverse Problem: Determine Well Count from Target Primary Production

Now, let's address the inverse problem: given a target primary production profile (from `pPR.csv`), we want to determine the well count vector required to achieve that production.

In [29]:
import numpy as np
import pandas as pd
from IPython.display import display, HTML

def generate_base_schedule_matrix(type_curve, num_months, ip_improvement_vector):
    """
    Generates a reusable base schedule matrix where each row represents a well vintage
    (the month a batch of wells is turned online) and each column represents a calendar timeline month.

    Parameters:
    type_curve (np.ndarray): The *normalized* production profile vector for a single standard well.
    num_months (int): Total number of months to model in the forecast timeline.
    ip_improvement_vector (np.ndarray): A vector of IP rates for each start_month (vintage).

    Returns:
    np.ndarray: A 2D matrix of shape (num_months, num_months) filled with shifted type curves,
                scaled by the appropriate IP rate for each vintage.
    """
    base_matrix = np.zeros((num_months, num_months))

    for start_month in range(num_months):
        # Calculate how many months are left in the forecast timeline from this start month forward
        months_remaining = num_months - start_month

        # Determine how much of the type curve fits into the remaining timeline window
        curve_length = min(len(type_curve), months_remaining)

        # Populate the row starting from the diagonal (the month the well goes online)
        # Scale the normalized type curve by the IP rate for that specific vintage
        base_matrix[start_month, start_month:start_month + curve_length] = type_curve[:curve_length] * ip_improvement_vector[start_month]

    return base_matrix

# Load the target primary production data
pPR_df = pd.read_csv("pPR.csv")

# Dictionary to store the calculated well count vectors for each region
calculated_well_counts = {}

# Iterate through each region to perform the inverse calculation
for region, group in df.groupby('Region'):
    # --- Re-calculate necessary components for the region (as done in the forward pass) ---

    # Extract the new well decline curve (normalized)
    normalized_new_well_decline_curve_values_original = np.zeros(num_months)
    new_well_decline_curve_row = group.loc[group['Name'] == 'newWellDC', month_cols]
    if not new_well_decline_curve_row.empty:
        normalized_new_well_decline_curve_values_original = pd.to_numeric(new_well_decline_curve_row.iloc[0], errors='coerce').fillna(0).values

    # Retrieve P1 and P1end for the current region
    ip_rate = 0.0
    ip_rate_row = factor_df.loc[factor_df['Region'] == region, 'P1']
    if not ip_rate_row.empty:
        ip_rate = ip_rate_row.values[0]

    p1_end_rate = 0.0
    p1_end_rate_row = factor_df.loc[factor_df['Region'] == region, 'P1end']
    if not p1_end_rate_row.empty:
        p1_end_rate = p1_end_rate_row.values[0]

    # Generate IP Improvement Vector
    ip_improvement_vector = np.linspace(ip_rate, p1_end_rate, num_months)

    # Generate the base production matrix (which includes IP improvement per vintage)
    base_prod_matrix = generate_base_schedule_matrix(normalized_new_well_decline_curve_values_original, num_months, ip_improvement_vector)

    # Calculate legacy production values (M1 to M24)
    m0_value = 0.0
    m0_series_row = factor_df.loc[factor_df['Region'] == region, 'M0'] * 1000
    if not m0_series_row.empty:
        m0_value = pd.to_numeric(m0_series_row, errors='coerce').fillna(0).iloc[0]

    legacy_decline_curve_values = np.zeros(num_months)
    legacy_decline_curve_row = group.loc[group['Name'] == 'legacyDC', month_cols]
    if not legacy_decline_curve_row.empty:
        legacy_decline_curve_values = pd.to_numeric(legacy_decline_curve_row.iloc[0], errors='coerce').fillna(0).values * m0_value

    # --- Get target total primary production from pPR_df for M1 to M24 ---
    # Check if the region column exists in pPR_df
    if region not in pPR_df.columns:
        print(f"Warning: No target production column found for region {region} in pPR.csv. Skipping.")
        calculated_well_counts[region] = np.full(num_months, np.nan) # Fill with NaN if column missing
        continue

    # Extract the target production values for the current region (M1 to M24)
    # Assuming the first 'num_months' rows correspond to M1 to M24 in pPR.csv
    # Multiply by 1000 to convert from '000 units to full units, matching the legacy production scale
    target_production_values_M1_to_M24 = pd.to_numeric(pPR_df[region].iloc[:num_months], errors='coerce').fillna(0).values * 1000

    # Calculate the desired new well production profile (M1 to M24)
    # desired_new_well_production = target_production (M1-M24) - legacy_decline_curve_values (M1-M24)
    desired_new_well_production = target_production_values_M1_to_M24 - legacy_decline_curve_values

    # Ensure desired_new_well_production does not have negative values, as that would imply negative well counts.
    # If target is less than legacy, it implies reducing wells, which this model cannot represent by positive well counts.
    # We will clamp to 0 and note this limitation for now.
    desired_new_well_production = np.maximum(0, desired_new_well_production)

    # Solve the linear system: A @ W = P_new_total_desired
    # Where A_matrix is the transpose of base_prod_matrix (base_prod_matrix[i, k] is the contribution
    # of well vintage 'i' to calendar month 'k' production, already scaled by IP improvement).
    A_matrix = base_prod_matrix.T

    try:
        # The result 'well_count_vector_calculated' represents the number of new wells
        # starting in each month (M1 to M24) that are needed.
        well_count_vector_calculated = np.linalg.solve(A_matrix, desired_new_well_production)
        # Ensure well counts are non-negative and integer values
        calculated_well_counts[region] = np.maximum(0, well_count_vector_calculated).round(0).astype(int)
    except np.linalg.LinAlgError as e:
        print(f"Could not solve for well counts for region {region}: {e}. This might happen if the production matrix is singular or ill-conditioned. Skipping this region.")
        calculated_well_counts[region] = np.full(num_months, np.nan) # Fill with NaN if cannot solve

# Create a DataFrame for the calculated well counts
calculated_well_counts_df = pd.DataFrame(calculated_well_counts, index=month_cols)
calculated_well_counts_df['L48'] = calculated_well_counts_df.sum(axis=1) # Sum across regions for L48 total

print("\n--- Calculated Well Count Vector by Vintage (M1 to M24) ---")
display(calculated_well_counts_df.style.format("{:,.0f}"))


--- Calculated Well Count Vector by Vintage (M1 to M24) ---


,Appalachia,Bakken,EagleFord,Haynesville,Permian,R48,L48
M1,41,0,0,41,0,0,82
M2,112,243,269,97,"1,272",410,"2,403"
M3,0,0,7,0,0,0,7
M4,269,430,278,6,"1,037",633,"2,653"
M5,0,0,73,27,75,0,175
M6,350,707,148,50,988,714,"2,957"
M7,0,0,48,52,95,0,195
M8,184,"1,084",233,33,"1,133","1,209","3,876"
M9,0,0,63,21,0,0,84
M10,168,"1,368",196,31,"1,223","1,977","4,963"


In [30]:
monthly_gor_values
#gor_monthly_growth

array([12.5       , 12.52083333, 12.54166667, 12.5625    , 12.58333333,
       12.60416667, 12.625     , 12.64583333, 12.66666667, 12.6875    ,
       12.70833333, 12.72916667, 12.75      , 12.77083333, 12.79166667,
       12.8125    , 12.83333333, 12.85416667, 12.875     , 12.89583333,
       12.91666667, 12.9375    , 12.95833333, 12.97916667, 13.        ])

In [31]:
new_well_decline_curve_values

array([165.908, 380.   , 361.532, 323.494, 284.088, 249.166, 219.26 ,
       195.548, 173.052, 156.712, 142.69 , 130.796, 121.448, 113.506,
       105.944,  99.028,  93.328,  88.274,  83.22 ,  78.28 ,  74.594,
        72.086,  68.894,  66.196])

In [32]:
print('New Well Production Matrix (BOE/day):')
new_well_prod_df = pd.DataFrame(new_well_production_matrix)
display(new_well_prod_df.style.format("{:,.0f}"))
print("\n")

New Well Production Matrix (BOE/day):


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23
0,"24,720","56,620","53,868","48,201","42,329","37,126","32,670","29,137","25,785","23,350","21,261","19,489","18,096","16,912","15,786","14,755","13,906","13,153","12,400","11,664","11,115","10,741","10,265","9,863"
1,0,"30,029","68,780","65,437","58,552","51,420","45,099","39,686","35,394","31,322","28,365","25,827","23,674","21,982","20,545","19,176","17,924","16,892","15,978","15,063","14,169","13,502","13,048","12,470"
2,0,0,"30,859","70,680","67,245","60,170","52,840","46,345","40,782","36,372","32,188","29,148","26,540","24,328","22,589","21,112","19,706","18,419","17,359","16,419","15,479","14,560","13,874","13,408"
3,0,0,0,"31,025","71,060","67,606","60,493","53,124","46,594","41,002","36,567","32,361","29,305","26,683","24,459","22,711","21,226","19,812","18,518","17,452","16,507","15,562","14,638","13,949"
4,0,0,0,0,"30,361","69,540","66,160","59,199","51,988","45,597","40,125","35,785","31,669","28,678","26,112","23,936","22,225","20,772","19,388","18,122","17,079","16,154","15,229","14,325"
5,0,0,0,0,0,"22,398","51,300","48,807","43,672","38,352","33,637","29,600","26,399","23,362","21,156","19,263","17,657","16,395","15,323","14,302","13,369","12,599","11,917","11,235"
6,0,0,0,0,0,0,"31,688","72,580","69,053","61,787","54,261","47,591","41,879","37,350","33,053","29,932","27,254","24,982","23,197","21,680","20,235","18,914","17,826","16,860"
7,0,0,0,0,0,0,0,"33,348","76,380","72,668","65,022","57,102","50,082","44,071","39,305","34,783","31,499","28,681","26,290","24,411","22,815","21,295","19,905","18,759"
8,0,0,0,0,0,0,0,0,"35,504","81,320","77,368","69,228","60,795","53,322","46,922","41,847","37,033","33,536","30,536","27,990","25,990","24,290","22,672","21,192"
9,0,0,0,0,0,0,0,0,0,"39,652","90,820","86,406","77,315","67,897","59,551","52,403","46,736","41,359","37,454","34,103","31,260","29,026","27,128","25,321"


# Task
The task is to enhance the new well production calculation by incorporating an IP (Initial Production) improvement vector. This involves generating a monthly IP improvement vector for each region based on 'P1' and 'P1end' values, and then integrating this vector into the new well decline curve calculation. Finally, the proposed method for modeling the IP improvement vector will be summarized.

## Understand IP Rate Modeling

### Subtask:
Examine the current code where the IP rate is applied and identify the section to be modified for incorporating the IP improvement vector.


```markdown
The existing code applies the IP rate as follows:

1.  **Retrieval of IP Rate**: For each region, the `ip_rate` is extracted from the `FactorDataT.csv` DataFrame. This happens in the line:
    ```python
    ip_rate_row = factor_df.loc[factor_df['Region'] == region, 'P1']
    if not ip_rate_row.empty:
        ip_rate = ip_rate_row.values[0]
    ```
    Here, `P1` from `factor_df` is assigned as the `ip_rate`.

2.  **Application of IP Rate**: The retrieved `ip_rate` is then used to scale the `new_well_decline_curve_values` (which are normalized decline curves) for the current region. This multiplication occurs in the line:
    ```python
    new_well_decline_curve_values = new_well_decline_curve_values * ip_rate
    ```
    This effectively converts the normalized decline curve into an absolute production rate based on the initial production (IP) rate `P1`.

To incorporate an IP improvement vector, this specific line `new_well_decline_curve_values = new_well_decline_curve_values * ip_rate` will need to be modified to use a vector of IP rates instead of a single scalar `ip_rate`.
```

## Retrieve P1 and P1end

### Subtask:
For each region, retrieve the 'P1' (start IP rate) and 'P1end' (exit IP rate) values from the 'FactorDataT.csv' DataFrame.


**Reasoning**:
The subtask requires retrieving 'P1' and 'P1end' for each region. The 'P1' value is already being retrieved, so I need to retrieve 'P1end' within the regional loop from the `factor_df`.



In [33]:
import numpy as np
import pandas as pd
from IPython.display import display, HTML

# The generate_arps_profile function is not used for the new well production calculation.

# Define the start date as a variable
START_DATE_STR = '2023-12-01' # This represents the M0 month, e.g., 'YYYY-MM-DD'
timeline_length = 24 # Number of months to forecast



# 1. Load the CSV files
df = pd.read_csv("CurveData.csv")
factor_df = pd.read_csv("FactorDataT.csv")
well_count_df = pd.read_csv("WellCount.csv")

def generate_base_schedule_matrix(type_curve, num_months, ip_improvement_vector):
    """
    Generates a reusable base schedule matrix where each row represents a well vintage
    (the month a batch of wells is turned online) and each column represents a calendar timeline month.

    Parameters:
    type_curve (np.ndarray): The *normalized* production profile vector for a single standard well.
    num_months (int): Total number of months to model in the forecast timeline.
    ip_improvement_vector (np.ndarray): A vector of IP rates for each start_month (vintage).

    Returns:
    np.ndarray: A 2D matrix of shape (num_months, num_months) filled with shifted type curves,
                scaled by the appropriate IP rate for each vintage.
    """
    base_matrix = np.zeros((num_months, num_months))

    for start_month in range(num_months):
        # Calculate how many months are left in the forecast timeline from this start month forward
        months_remaining = num_months - start_month

        # Determine how much of the type curve fits into the remaining timeline window
        curve_length = min(len(type_curve), months_remaining)

        # Populate the row starting from the diagonal (the month the well goes online)
        # Scale the normalized type curve by the IP rate for that specific vintage
        base_matrix[start_month, start_month:start_month + curve_length] = type_curve[:curve_length] * ip_improvement_vector[start_month]

    return base_matrix



# Identify our timeline_length in months for the new decline curve (M1 through Mxx)
# Corrected to use uppercase 'M' for column names to match the CSV data
month_cols = [f'M{i+1}' for i in range(timeline_length)]
num_months = len(month_cols)

# Dictionary to hold the final forecast timelines for analysis
regional_forecasts = {}
regional_ip_improvement_vectors = {}

# 2. Process data dynamically by Region for new well production
for region, group in df.groupby('Region'):
    # Extract the new well decline curve (normalized) for the current region
    # Convert non-numeric values to 0 and default to zeros if 'newWellDC' is not found
    # Store the original normalized curve before any scaling
    normalized_new_well_decline_curve_values_original = np.zeros(num_months)
    legacy_decline_curve_values = np.zeros(num_months)

    new_well_decline_curve_row = group.loc[group['Name'] == 'newWellDC', month_cols]
    legacy_decline_curve_row = group.loc[group['Name'] == 'legacyDC', month_cols]

    if not new_well_decline_curve_row.empty:
        normalized_new_well_decline_curve_values_original = pd.to_numeric(new_well_decline_curve_row.iloc[0], errors='coerce').fillna(0).values
    if not legacy_decline_curve_row.empty:
        legacy_decline_curve_values = pd.to_numeric(legacy_decline_curve_row.iloc[0], errors='coerce').fillna(0).values

    # Get IP rate (P1) for the current region from FactorDataT.csv
    # Default to 0.0 if not found for the region
    ip_rate = 0.0
    ip_rate_row = factor_df.loc[factor_df['Region'] == region, 'P1']
    if not ip_rate_row.empty:
        ip_rate = ip_rate_row.values[0]

    # Retrieve P1end for the current region
    p1_end_rate = 0.0
    p1_end_rate_row = factor_df.loc[factor_df['Region'] == region, 'P1end']
    if not p1_end_rate_row.empty:
        p1_end_rate = p1_end_rate_row.values[0]

    # Generate IP Improvement Vector
    # Linearly interpolate between P1 and P1end over the timeline_length
    ip_improvement_vector = np.linspace(ip_rate, p1_end_rate, num_months)
    regional_ip_improvement_vectors[region] = ip_improvement_vector

    # The previous line `new_well_decline_curve_values = new_well_decline_curve_values * ip_rate` is removed.
    # The scaling will now happen inside generate_base_schedule_matrix for each vintage.

    # Get Well Count *vector* for the current region from WellCount.csv
    # This should be a vector of well counts for M1 to M24, where each element corresponds
    # to the well count for the vintage starting in that month.
    well_count_vector = np.zeros(num_months)
    well_count_row_full = well_count_df.loc[(well_count_df['Region'] == region) & (well_count_df['Name'] == 'wellCount'), month_cols]
    if not well_count_row_full.empty:
        well_count_vector = pd.to_numeric(well_count_row_full.iloc[0], errors='coerce').fillna(0).values

    # Generate the reusable Base Schedule Matrix template by passing the normalized curve and the IP improvement vector
    base_prod_matrix = generate_base_schedule_matrix(normalized_new_well_decline_curve_values_original, num_months, ip_improvement_vector)

    # Scale the base unit matrix by the scenario's well count column (well_count_vector) using broadcasting
    new_well_production_matrix = base_prod_matrix * well_count_vector[:, np.newaxis]


    # Get M0 production rate for the current region from FactorDataT.csv
    # Default to 0.0 if not found for the region
    m0_value = 0.0 # Initialize scalar M0
    m0_series_row = factor_df.loc[factor_df['Region'] == region, 'M0']*1000
    if not m0_series_row.empty:
        # Extract scalar value from the series for both calculation and prepending
        m0_value = pd.to_numeric(m0_series_row, errors='coerce').fillna(0).iloc[0]

    # Adjust normalized legacy decline values with M0 rate for the current region
    # Use the scalar m0_value for multiplication
    legacy_decline_curve_values = legacy_decline_curve_values * m0_value

    # Sum columns to get total timeline production for this region (timeline_length months)
    current_regional_forecast = legacy_decline_curve_values + np.sum(new_well_production_matrix, axis=0)

    # Prepend the M0 value and timeline_length months
    regional_forecasts[region] = np.insert(current_regional_forecast, 0, m0_value)





# 3. Combine results into a clean summary DataFrame
# Convert START_DATE_STR to datetime object
start_date_dt = pd.to_datetime(START_DATE_STR)

# Define the new index labels including the dynamic start month
initial_month_label = start_date_dt.strftime('%b_%Y')

# Generate dates for the subsequent months
dates_after_initial_month = pd.date_range(start=start_date_dt + pd.DateOffset(months=1), periods=num_months, freq='MS')
formatted_dates = [d.strftime('%b_%Y') for d in dates_after_initial_month]

new_index_labels = [initial_month_label] + formatted_dates

forecast_df = pd.DataFrame(regional_forecasts, index=new_index_labels)
forecast_df['L48'] = forecast_df.sum(axis=1)
forecast_df = forecast_df.astype(float)

print("Primary Commodity Forecast Summary:")

display(forecast_df.style.format("{:,.0f}"))
#print(forecast_df)

Primary Commodity Forecast Summary:


,Appalachia,Bakken,EagleFord,Haynesville,Permian,R48,L48
Dec_2023,"36,722,000","1,305,000","1,068,000","16,014,000","6,193,000","2,096,000","63,398,000"
Jan_2024,"36,625,105","1,297,516","1,069,418","15,973,921","6,195,689","2,086,555","63,248,204"
Feb_2024,"36,296,855","1,289,509","1,081,109","15,919,481","6,164,495","2,069,568","62,821,017"
Mar_2024,"35,877,812","1,274,072","1,112,342","15,601,247","6,142,103","2,060,896","62,068,472"
Apr_2024,"35,656,914","1,251,132","1,143,298","15,177,655","6,167,159","2,055,612","61,451,771"
May_2024,"35,597,285","1,238,954","1,151,076","14,889,568","6,258,406","2,049,605","61,184,894"
Jun_2024,"35,374,304","1,231,109","1,135,046","14,721,137","6,315,543","2,036,731","60,813,869"
Jul_2024,"35,341,688","1,225,766","1,145,498","14,752,926","6,324,480","2,022,939","60,813,297"
Aug_2024,"35,262,345","1,220,750","1,164,703","14,654,629","6,323,450","2,023,730","60,649,608"
Sep_2024,"34,977,231","1,230,217","1,173,309","14,403,924","6,315,047","2,030,139","60,129,867"


In [9]:
regional_ip_improvement_vectors[region]

array([380.        , 381.30434783, 382.60869565, 383.91304348,
       385.2173913 , 386.52173913, 387.82608696, 389.13043478,
       390.43478261, 391.73913043, 393.04347826, 394.34782609,
       395.65217391, 396.95652174, 398.26086957, 399.56521739,
       400.86956522, 402.17391304, 403.47826087, 404.7826087 ,
       406.08695652, 407.39130435, 408.69565217, 410.        ])

In [34]:
print('New Single Well Production Matrix (BOE/day):')
new_single_well_prod_df = pd.DataFrame(base_prod_matrix)
display(new_single_well_prod_df.style.format("{:,.0f}"))
print("\n")

New Single Well Production Matrix (BOE/day):


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23
0,166,380,362,323,284,249,219,196,173,157,143,131,121,114,106,99,93,88,83,78,75,72,69,66
1,0,166,381,363,325,285,250,220,196,174,157,143,131,122,114,106,99,94,89,84,79,75,72,69
2,0,0,167,383,364,326,286,251,221,197,174,158,144,132,122,114,107,100,94,89,84,79,75,73
3,0,0,0,168,384,365,327,287,252,222,198,175,158,144,132,123,115,107,100,94,89,84,79,75
4,0,0,0,0,168,385,366,328,288,253,222,198,175,159,145,133,123,115,107,100,95,89,84,79
5,0,0,0,0,0,169,387,368,329,289,253,223,199,176,159,145,133,124,115,108,101,95,90,85
6,0,0,0,0,0,0,169,388,369,330,290,254,224,200,177,160,146,133,124,116,108,101,95,90
7,0,0,0,0,0,0,0,170,389,370,331,291,255,225,200,177,160,146,134,124,116,108,101,96
8,0,0,0,0,0,0,0,0,170,390,371,332,292,256,225,201,178,161,147,134,125,117,109,102
9,0,0,0,0,0,0,0,0,0,171,392,373,333,293,257,226,202,178,162,147,135,125,117,109


### Inverse Problem: Determine Well Count from Target Primary Production (Allowing Negative Well Counts)

Let's re-run the inverse problem, but this time allowing for the possibility of negative well counts. This might be interpreted as decommissioning or shutting in wells if the target production is significantly lower than the legacy production.

In [35]:
# Load the target primary production data
pPR_df = pd.read_csv("pPR.csv")

# Dictionary to store the calculated well count vectors for each region
calculated_well_counts_neg_allowed = {}

# Iterate through each region to perform the inverse calculation
for region, group in df.groupby('Region'):
    # --- Re-calculate necessary components for the region (as done in the forward pass) ---

    # Extract the new well decline curve (normalized)
    normalized_new_well_decline_curve_values_original = np.zeros(num_months)
    new_well_decline_curve_row = group.loc[group['Name'] == 'newWellDC', month_cols]
    if not new_well_decline_curve_row.empty:
        normalized_new_well_decline_curve_values_original = pd.to_numeric(new_well_decline_curve_row.iloc[0], errors='coerce').fillna(0).values

    # Retrieve P1 and P1end for the current region
    ip_rate = 0.0
    ip_rate_row = factor_df.loc[factor_df['Region'] == region, 'P1']
    if not ip_rate_row.empty:
        ip_rate = ip_rate_row.values[0]

    p1_end_rate = 0.0
    p1_end_rate_row = factor_df.loc[factor_df['Region'] == region, 'P1end']
    if not p1_end_rate_row.empty:
        p1_end_rate = p1_end_rate_row.values[0]

    # Generate IP Improvement Vector
    ip_improvement_vector = np.linspace(ip_rate, p1_end_rate, num_months)

    # Generate the base production matrix (which includes IP improvement per vintage)
    base_prod_matrix = generate_base_schedule_matrix(normalized_new_well_decline_curve_values_original, num_months, ip_improvement_vector)

    # Calculate legacy production values (M1 to M24)
    m0_value = 0.0
    m0_series_row = factor_df.loc[factor_df['Region'] == region, 'M0'] * 1000
    if not m0_series_row.empty:
        m0_value = pd.to_numeric(m0_series_row, errors='coerce').fillna(0).iloc[0]

    legacy_decline_curve_values = np.zeros(num_months)
    legacy_decline_curve_row = group.loc[group['Name'] == 'legacyDC', month_cols]
    if not legacy_decline_curve_row.empty:
        legacy_decline_curve_values = pd.to_numeric(legacy_decline_curve_row.iloc[0], errors='coerce').fillna(0).values * m0_value

    # --- Get target total primary production from pPR_df for M1 to M24 ---
    # Check if the region column exists in pPR_df
    if region not in pPR_df.columns:
        print(f"Warning: No target production column found for region {region} in pPR.csv. Skipping.")
        calculated_well_counts_neg_allowed[region] = np.full(num_months, np.nan) # Fill with NaN if column missing
        continue

    # Extract the target production values for the current region (M1 to M24)
    # Assuming the first 'num_months' rows correspond to M1 to M24 in pPR.csv
    # Multiply by 1000 to convert from '000 units to full units, matching the legacy production scale
    target_production_values_M1_to_M24 = pd.to_numeric(pPR_df[region].iloc[:num_months], errors='coerce').fillna(0).values * 1000

    # Calculate the desired new well production profile (M1 to M24)
    desired_new_well_production = target_production_values_M1_to_M24 - legacy_decline_curve_values

    # Solve the linear system: A @ W = P_new_total_desired
    # Where A_matrix is the transpose of base_prod_matrix (base_prod_matrix[i, k] is the contribution
    # of well vintage 'i' to calendar month 'k' production, already scaled by IP improvement).
    A_matrix = base_prod_matrix.T

    try:
        # The result 'well_count_vector_calculated' represents the number of new wells
        # starting in each month (M1 to M24) that are needed.
        well_count_vector_calculated = np.linalg.solve(A_matrix, desired_new_well_production)
        # Allowing negative values, and rounding to nearest integer
        calculated_well_counts_neg_allowed[region] = well_count_vector_calculated.round(0).astype(int)
    except np.linalg.LinAlgError as e:
        print(f"Could not solve for well counts for region {region}: {e}. This might happen if the production matrix is singular or ill-conditioned. Skipping this region.")
        calculated_well_counts_neg_allowed[region] = np.full(num_months, np.nan) # Fill with NaN if cannot solve

# Create a DataFrame for the calculated well counts
calculated_well_counts_neg_allowed_df = pd.DataFrame(calculated_well_counts_neg_allowed, index=month_cols)
calculated_well_counts_neg_allowed_df['L48'] = calculated_well_counts_neg_allowed_df.sum(axis=1) # Sum across regions for L48 total

print("\n--- Calculated Well Count Vector by Vintage (M1 to M24) - Negative Counts Allowed ---")
display(calculated_well_counts_neg_allowed_df.style.format("{:,.0f}"))


--- Calculated Well Count Vector by Vintage (M1 to M24) - Negative Counts Allowed ---


,Appalachia,Bakken,EagleFord,Haynesville,Permian,R48,L48
M1,41,-556,0,41,-174,-644,"-1,292"
M2,112,"1,415",269,97,"1,580","1,879","5,352"
M3,-182,"-1,510",7,-31,-341,"-2,100","-4,157"
M4,269,"2,042",278,6,"1,293","3,172","7,060"
M5,-109,"-2,392",73,27,-153,"-3,679","-6,233"
M6,350,"3,029",148,50,"1,195","5,004","9,776"
M7,-155,"-3,484",48,52,-90,"-6,099","-9,728"
M8,184,"4,429",233,33,"1,300","8,463","14,642"
M9,-45,"-5,038",63,21,-199,"-10,517","-15,715"
M10,168,"6,149",196,31,"1,354","14,292","22,190"


### Comparison of Well Count Plans: Original vs. Negative Counts Allowed

Below, we compare the two scenarios:

1.  **Original Well Count Plan**: This plan enforced a non-negative constraint on the `desired_new_well_production` (clamping values to 0) and subsequently on the calculated well counts. This reflects a scenario where new well additions are the only mechanism to meet target production, and reducing production below legacy levels for new wells is not explicitly modeled.

2.  **Negative Well Counts Allowed Scenario**: This plan removes the non-negative constraint on `desired_new_well_production` and allows the calculated well counts to be negative. Negative well counts can be interpreted as the need to 'decommission' or 'shut-in' wells to meet a target production lower than the legacy production in a given month. This provides a more direct mathematical solution to the inverse problem without an artificial floor on new well activity.

#### Original Calculated Well Counts (Non-negative constrained)

In [36]:
display(calculated_well_counts_df.style.format("{:,.0f}"))

,Appalachia,Bakken,EagleFord,Haynesville,Permian,R48,L48
M1,41,0,0,41,0,0,82
M2,112,243,269,97,"1,272",410,"2,403"
M3,0,0,7,0,0,0,7
M4,269,430,278,6,"1,037",633,"2,653"
M5,0,0,73,27,75,0,175
M6,350,707,148,50,988,714,"2,957"
M7,0,0,48,52,95,0,195
M8,184,"1,084",233,33,"1,133","1,209","3,876"
M9,0,0,63,21,0,0,84
M10,168,"1,368",196,31,"1,223","1,977","4,963"


#### Calculated Well Counts (Negative Counts Allowed)

In [37]:
display(calculated_well_counts_neg_allowed_df.style.format("{:,.0f}"))

,Appalachia,Bakken,EagleFord,Haynesville,Permian,R48,L48
M1,41,-556,0,41,-174,-644,"-1,292"
M2,112,"1,415",269,97,"1,580","1,879","5,352"
M3,-182,"-1,510",7,-31,-341,"-2,100","-4,157"
M4,269,"2,042",278,6,"1,293","3,172","7,060"
M5,-109,"-2,392",73,27,-153,"-3,679","-6,233"
M6,350,"3,029",148,50,"1,195","5,004","9,776"
M7,-155,"-3,484",48,52,-90,"-6,099","-9,728"
M8,184,"4,429",233,33,"1,300","8,463","14,642"
M9,-45,"-5,038",63,21,-199,"-10,517","-15,715"
M10,168,"6,149",196,31,"1,354","14,292","22,190"


### Inverse Problem: Determine Well Count from Target Primary Production

Now, let's address the inverse problem: given a target primary production profile (from `pPR.csv`), we want to determine the well count vector required to achieve that production.

In [38]:
# Load the target primary production data
pPR_df = pd.read_csv("pPR.csv")

# Dictionary to store the calculated well count vectors for each region
calculated_well_counts = {}

# Iterate through each region to perform the inverse calculation
for region, group in df.groupby('Region'):
    # --- Re-calculate necessary components for the region (as done in the forward pass) ---

    # Extract the new well decline curve (normalized)
    normalized_new_well_decline_curve_values_original = np.zeros(num_months)
    new_well_decline_curve_row = group.loc[group['Name'] == 'newWellDC', month_cols]
    if not new_well_decline_curve_row.empty:
        normalized_new_well_decline_curve_values_original = pd.to_numeric(new_well_decline_curve_row.iloc[0], errors='coerce').fillna(0).values

    # Retrieve P1 and P1end for the current region
    ip_rate = 0.0
    ip_rate_row = factor_df.loc[factor_df['Region'] == region, 'P1']
    if not ip_rate_row.empty:
        ip_rate = ip_rate_row.values[0]

    p1_end_rate = 0.0
    p1_end_rate_row = factor_df.loc[factor_df['Region'] == region, 'P1end']
    if not p1_end_rate_row.empty:
        p1_end_rate = p1_end_rate_row.values[0]

    # Generate IP Improvement Vector
    ip_improvement_vector = np.linspace(ip_rate, p1_end_rate, num_months)

    # Generate the base production matrix (which includes IP improvement per vintage)
    base_prod_matrix = generate_base_schedule_matrix(normalized_new_well_decline_curve_values_original, num_months, ip_improvement_vector)

    # Calculate legacy production values (M1 to M24)
    m0_value = 0.0
    m0_series_row = factor_df.loc[factor_df['Region'] == region, 'M0'] * 1000
    if not m0_series_row.empty:
        m0_value = pd.to_numeric(m0_series_row, errors='coerce').fillna(0).iloc[0]

    legacy_decline_curve_values = np.zeros(num_months)
    legacy_decline_curve_row = group.loc[group['Name'] == 'legacyDC', month_cols]
    if not legacy_decline_curve_row.empty:
        legacy_decline_curve_values = pd.to_numeric(legacy_decline_curve_row.iloc[0], errors='coerce').fillna(0).values * m0_value

    # --- Get target total primary production from pPR_df for M1 to M24 ---
    # Check if the region column exists in pPR_df
    if region not in pPR_df.columns:
        print(f"Warning: No target production column found for region {region} in pPR.csv. Skipping.")
        calculated_well_counts[region] = np.full(num_months, np.nan) # Fill with NaN if column missing
        continue

    # Extract the target production values for the current region (M1 to M24)
    # Assuming the first 'num_months' rows correspond to M1 to M24 in pPR.csv
    # Multiply by 1000 to convert from '000 units to full units, matching the legacy production scale
    target_production_values_M1_to_M24 = pd.to_numeric(pPR_df[region].iloc[:num_months], errors='coerce').fillna(0).values * 1000

    # Calculate the desired new well production profile (M1 to M24)
    # desired_new_well_production = target_production (M1-M24) - legacy_decline_curve_values (M1-M24)
    desired_new_well_production = target_production_values_M1_to_M24 - legacy_decline_curve_values

    # Ensure desired_new_well_production does not have negative values, as that would imply negative well counts.
    # If target is less than legacy, it implies reducing wells, which this model cannot represent by positive well counts.
    # We will clamp to 0 and note this limitation for now.
    desired_new_well_production = np.maximum(0, desired_new_well_production)

    # Solve the linear system: A @ W = P_new_total_desired
    # Where A_matrix is the transpose of base_prod_matrix (base_prod_matrix[i, k] is the contribution
    # of well vintage 'i' to calendar month 'k' production, already scaled by IP improvement).
    A_matrix = base_prod_matrix.T

    try:
        # The result 'well_count_vector_calculated' represents the number of new wells
        # starting in each month (M1 to M24) that are needed.
        well_count_vector_calculated = np.linalg.solve(A_matrix, desired_new_well_production)
        # Ensure well counts are non-negative and integer values
        calculated_well_counts[region] = np.maximum(0, well_count_vector_calculated).round(0).astype(int)
    except np.linalg.LinAlgError as e:
        print(f"Could not solve for well counts for region {region}: {e}. This might happen if the production matrix is singular or ill-conditioned. Skipping this region.")
        calculated_well_counts[region] = np.full(num_months, np.nan) # Fill with NaN if cannot solve

# Create a DataFrame for the calculated well counts
calculated_well_counts_df = pd.DataFrame(calculated_well_counts, index=month_cols)
calculated_well_counts_df['L48'] = calculated_well_counts_df.sum(axis=1) # Sum across regions for L48 total

print("\n--- Calculated Well Count Vector by Vintage (M1 to M24) ---")
display(calculated_well_counts_df.style.format("{:,.0f}"))


--- Calculated Well Count Vector by Vintage (M1 to M24) ---


,Appalachia,Bakken,EagleFord,Haynesville,Permian,R48,L48
M1,41,0,0,41,0,0,82
M2,112,243,269,97,"1,272",410,"2,403"
M3,0,0,7,0,0,0,7
M4,269,430,278,6,"1,037",633,"2,653"
M5,0,0,73,27,75,0,175
M6,350,707,148,50,988,714,"2,957"
M7,0,0,48,52,95,0,195
M8,184,"1,084",233,33,"1,133","1,209","3,876"
M9,0,0,63,21,0,0,84
M10,168,"1,368",196,31,"1,223","1,977","4,963"


In [21]:
calculated_well_counts_df.to_csv('calculated_well_counts.csv')
calculated_well_counts_neg_allowed_df.to_csv('calculated_well_counts_neg_allowed.csv')

### Inverse Problem: Determine GOR Ratios from Target Non-Primary Production

Now, let's address another inverse problem: given a target non-primary production profile (from `npPR.csv`), we want to determine the GOR ratio vector required to achieve that production for each month.

In [39]:
# Load the target non-primary production data
npPR_df = pd.read_csv("npPR.csv")

# Dictionary to store the calculated GOR ratio vectors for each region
calculated_gor_ratios = {}

# Iterate through each region to perform the inverse calculation
for region in regional_forecasts.keys():
    # Skip L48 as it's a total, not a region for individual GOR calculation
    if region == 'L48':
        continue

    # Get primary production for the current region
    primary_production = forecast_df[region].iloc[1:].values # Exclude M0 as npPR.csv starts from M1

    # Check if the region column exists in npPR_df
    if region not in npPR_df.columns:
        print(f"Warning: No target non-primary production column found for region {region} in npPR.csv. Skipping.")
        calculated_gor_ratios[region] = np.full(num_months, np.nan) # Fill with NaN if column missing
        continue

    # Extract the target non-primary production values for the current region (M1 to M24)
    # Assuming the first 'num_months' rows correspond to M1 to M24 in npPR.csv
    target_non_primary_production = pd.to_numeric(npPR_df[region].iloc[:num_months], errors='coerce').fillna(0).values * 1000

    # Perform inverse GOR calculation
    monthly_gor_calculated = np.zeros(num_months)
    if region in GAS_REGIONS: # Primary is gas, non-primary is oil. Oil = Gas / GOR => GOR = Gas / Oil
        # Ensure target_non_primary_production is not zero to avoid division by zero
        monthly_gor_calculated = np.where(target_non_primary_production != 0, primary_production / target_non_primary_production, np.nan)
    else: # Primary is oil, non-primary is gas. Gas = Oil * GOR => GOR = Gas / Oil
        # Ensure primary_production is not zero to avoid division by zero
        monthly_gor_calculated = np.where(primary_production != 0, target_non_primary_production / primary_production, np.nan)

    calculated_gor_ratios[region] = monthly_gor_calculated

# Create a DataFrame for the calculated GOR ratios
calculated_gor_ratios_df = pd.DataFrame(calculated_gor_ratios, index=month_cols)

print("\n--- Calculated GOR Ratios (Mcf/bbl or bbl/Mcf depending on primary) ---")
display(calculated_gor_ratios_df.style.format("{:,.2f}"))


--- Calculated GOR Ratios (Mcf/bbl or bbl/Mcf depending on primary) ---


,Appalachia,Bakken,EagleFord,Haynesville,Permian,R48
M1,283.92,2.35,6.86,665.58,3.84,12.28
M2,283.57,2.65,6.98,612.29,4.00,12.77
M3,271.80,2.70,6.73,600.05,4.09,12.61
M4,264.13,2.82,6.34,607.11,4.08,12.60
M5,259.83,2.88,6.66,620.40,4.01,12.66
M6,250.88,2.88,6.51,640.05,4.10,12.59
M7,257.97,2.88,6.44,614.71,4.15,12.70
M8,248.33,2.94,6.31,637.16,4.23,12.58
M9,231.64,2.94,6.25,600.16,4.25,12.48
M10,238.59,2.80,6.48,572.39,4.30,12.36


## Generate IP Improvement Vector

### Subtask:
Propose a method to generate a monthly IP improvement vector for each region by linearly interpolating between the 'P1' and 'P1end' values over the 'timeline_length'. This vector will define the changing IP rate for each month of the forecast.


# Task
The task is to enhance the new well production calculation by incorporating an IP (Initial Production) improvement vector. This involves generating a monthly IP improvement vector for each region based on 'P1' and 'P1end' values, and then integrating this vector into the new well decline curve calculation. Finally, the proposed method for modeling the IP improvement vector will be summarized.

## Integrate IP Vector Correctly

### Subtask:
Modify the `generate_base_schedule_matrix` function to accept the `ip_improvement_vector` as an additional argument. Inside this function, for each `start_month` (representing a well vintage), scale the `normalized_new_well_decline_curve_values` by the corresponding IP rate from the `ip_improvement_vector` (i.e., `ip_improvement_vector[start_month]`). This ensures that each well vintage is scaled by its unique initial IP rate based on the improvement vector. Also, ensure that the `new_well_decline_curve_values` passed to `generate_base_schedule_matrix` is the original normalized version, not previously scaled by a single `ip_rate`.


## Final Task

### Subtask:
Provide a summary of the implemented method for modeling the IP improvement vector and present the updated production forecast.


# Task
The task is to enhance the new well production calculation by incorporating an IP (Initial Production) improvement vector. This involves generating a monthly IP improvement vector for each region based on 'P1' and 'P1end' values, and then integrating this vector into the new well decline curve calculation. Finally, the proposed method for modeling the IP improvement vector will be summarized.

## Final Task

### Subtask:
Provide a summary of the implemented method for modeling the IP improvement vector and present the updated production forecast.
